# Week 2 Step 11: Saving the three models

I am saving each chosen model **with its preprocessor** so Week 3 can load a `.pkl` and score new rows. I already picked the models on train/test splits. For the copilot I refit on all 8,000 customers.

What I am serializing:
- churn: balanced logistic regression, threshold 0.46 from the training PR curve
- segmentation: K-Means with K=4, median impute + StandardScaler
- CLV: random forest trained on `log1p(CLV)`, predictions converted back with `expm1`

I also store `churn_risk_score` for every customer, plus the cluster label, because the dashboard will need both.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.save_models import save_all

info = save_all(PROJECT_ROOT)
print(json.dumps(info, indent=2))
print("files:", sorted(p.name for p in (PROJECT_ROOT / "models").iterdir()))

{
  "n_customers": 8000,
  "models_dir": "c:\\Users\\saksh\\OneDrive - University of Keele\\Documents\\Netflix_subscription_project\\models",
  "features_path": "c:\\Users\\saksh\\OneDrive - University of Keele\\Documents\\Netflix_subscription_project\\data\\processed\\customer_features.csv",
  "metadata_path": "c:\\Users\\saksh\\OneDrive - University of Keele\\Documents\\Netflix_subscription_project\\models\\model_metadata.json",
  "churn_risk_mean": 0.46224062162230956
}
files: ['churn_model.pkl', 'clv_model.pkl', 'model_metadata.json', 'satisfaction_model.pkl', 'segmentation_model.pkl']


A model that only loads in this notebook is not saved. I am checking from a **fresh Python process** that each `.pkl` can score a few rows of `customer_features.csv`.

In [2]:
import subprocess

result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "src" / "models" / "reload_check.py")],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

reload ok
churn_risk_score: [0.461, 0.772, 0.552, 0.495, 0.856]
cluster: [3, 2, 2, 0, 2]
clv_pred: [298.63, 36.99, 93.27, 219.82, 95.72]



In [3]:
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "customer_features.csv")
meta = json.loads((PROJECT_ROOT / "models" / "model_metadata.json").read_text(encoding="utf-8"))

print("threshold:", meta["churn_threshold"])
print("cluster names:", meta["cluster_names"])
print("churn_risk_score missing:", int(df["churn_risk_score"].isna().sum()))
print("mean churn_risk_score:", round(df["churn_risk_score"].mean(), 3))
display(df[["customer_id", "churned", "churn_risk_score", "cluster", "cluster_name"]].head())

churn = joblib.load(PROJECT_ROOT / "models" / "churn_model.pkl")
sample = df.head(3).drop(columns=["churn_risk_score", "cluster", "cluster_name", "churned"], errors="ignore")
print("fresh predict on 3 rows:", churn.predict_proba(sample[meta["churn_input_columns"]])[:, 1].round(3))

threshold: 0.46
cluster names: {'0': 'Long-tenure payment friction', '1': 'Dormant at-risk', '2': 'Newer active signups', '3': 'High-value loyal'}
churn_risk_score missing: 0
mean churn_risk_score: 0.462


,customer_id,churned,churn_risk_score,cluster,cluster_name
0,CUST000744,False,0.460950,3,High-value loyal
1,CUST001502,False,0.771609,2,Newer active signups
2,CUST005383,False,0.551748,2,Newer active signups
3,CUST004708,True,0.494651,0,Long-tenure payment friction
4,CUST005877,False,0.855571,2,Newer active signups


fresh predict on 3 rows: [0.461 0.772 0.552]
